# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [1]:
%load_ext dotenv
%dotenv ../05_src/.secrets

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

## Imports 
<b><i>Separated imports for manageability.<i></b>

In [2]:
# Imports
import os, json
from openai import OpenAI
from pydantic import BaseModel
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

## Load API Key

In [3]:
# Load API Key

client = OpenAI(base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
api_key='any value',
default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')})

## Load PDF 
<i>(pdf must exist in "documents/ folder"). </i>
<i>
- I used Chunking to handle loading .pdf documents to handle documents of any size and to save on token costs.
- I also chunked the PDF but concatenated the chunks back together for simplicity as this suits my use case.
- I did not use "overlap" in the chunking since I am just concatenating everything back together anyway.
</i>

In [4]:
# Load PDF
loader = PyPDFLoader("documents/managing_oneself.pdf")
docs = loader.load()

 # Create text from docs
text = "\n".join(p.page_content for p in docs)
print(f"Loaded {len(docs)} page(s). Total characters: {len(text):,}")

# chunk for robustness (but reassemble for single summary)
splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,
    )
chunks = splitter.split_text(text)
text = "\n".join(chunks)  # Reassemble with clear boundaries
print(f"Split into {len(chunks)} chunk(s)")

document_text = text

Loaded 13 page(s). Total characters: 51,451
Split into 29 chunk(s)


## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


## Call OpenAPI Model (gpt-4o-mini)


<i>This code sends an article to the model, asks for a structured summary and metadata, and prints the result as JSON.

This is how the code works:

- Defines two Pydantic models:

   - ArticleSummary with fields: Author, Title, Relevance, Summary, Tone.
   
   - ArticleSummaryTokensUsed that extends ArticleSummary and adds InputTokens and OutputTokens (optional ints).

   This was done to separate internal tracking in the API.

- I used the chat completions API:

   - A clear separation of developer + user roles lets us separate instructions from content.
   - It contains built-in token tracking.

- Constructs a dictionary with the extracted fields plus token usage and prints it as pretty JSON.
</i> 


In [5]:
class ArticleSummary(BaseModel):
    Author: str
    Title: str
    Relevance: str
    Summary: str
    Tone: str

class ArticleSummaryTokensUsed(ArticleSummary):
    InputTokens: int | None = None
    OutputTokens: int | None = None

DEVELOPER_PROMPT = """
You are an expert academic summarization assistant.

1. Author: extract the author.
2. Title: extract the title.
3. Relevance: write one paragraph on why this article matters for an AI professional's
   professional development.
3. Relevance: write a statement, no longer than one paragraph, that explains why this 
   article is relevant for an AI professional in their professional development
4. Summary: write a concise and succinct summary no longer than 1000 tokens.
5. Tone: set the tone of the summary to "Formal Academic Writing".
"""

USER_PROMPT = f"""Article content:

{document_text}
"""

response = client.chat.completions.parse(
    model="gpt-4o-mini",
    temperature=0.2,
    messages=[
        {"role": "developer", "content": DEVELOPER_PROMPT},
        {"role": "user", "content": USER_PROMPT},
    ],
    response_format=ArticleSummary,
)

parsed = response.choices[0].message.parsed
parsed_output = ArticleSummaryTokensUsed(
    **parsed.model_dump(),
    InputTokens=response.usage.prompt_tokens,
    OutputTokens=response.usage.completion_tokens,
)


formatted = {
"Author": parsed_output.Author,
"Title": parsed_output.Title,
"Relevance": parsed_output.Relevance,
"Summary": parsed_output.Summary,
"Tone": parsed_output.Tone,
"InputTokens": parsed_output.InputTokens,
"OutputTokens": parsed_output.OutputTokens,
}
print(json.dumps(formatted, ensure_ascii=False, indent=2))

{
  "Author": "Peter F. Drucker",
  "Title": "Managing Oneself",
  "Relevance": "This article is crucial for AI professionals as it emphasizes the importance of self-awareness in career development. In a rapidly evolving field like artificial intelligence, understanding one's strengths, weaknesses, and values is essential for navigating career paths and making informed decisions about skill development and job opportunities. By applying Drucker's principles, AI professionals can enhance their effectiveness, adapt to changing environments, and contribute meaningfully to their organizations.",
  "Summary": "In 'Managing Oneself', Peter F. Drucker argues that success in the knowledge economy hinges on self-awareness and personal responsibility. He posits that individuals must act as their own chief executive officers, taking charge of their careers and understanding their strengths, weaknesses, values, and preferred work styles. Drucker introduces the concept of feedback analysis as a met

## Save Summary for Part 2

**summary_text contains the document summary

**document_text contains the original document

In [6]:
# Save summary
summary_data = formatted
summary_text = parsed_output.Summary

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [7]:
# Imports
import os
import json
from openai import OpenAI
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from deepeval import evaluate
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval.models import GPTModel
from IPython.display import display, Markdown


In [8]:
# Load API Key

client = OpenAI(base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
api_key='any value',
default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')})

In [9]:
import json

# Load cached summary from Part 1
with open('summary_output.json', 'r') as f:
    formatted = json.load(f)
    summary_text = formatted['Summary']

print(f"Summary loaded from Part 1: {len(summary_text)} characters")

Summary loaded from Part 1: 1417 characters


In [10]:
# Initialize GPT Model for DeepEval
model = GPTModel(
    model="gpt-4o-mini",
    temperature=0,
)

The original Summarization Metric will be captured in variables for use in
later comparisons.
- summarization_score
- summarization_reason

I also set the Threshold value to the recommended threshold per the GEval documentation.

In [11]:
# 1. SUMMARIZATION G-EVAL METRIC with bespoke assessment questions
summarization_metric = GEval(
    name="Summarization",
    evaluation_steps=[
        "Does the summary capture the main thesis or central argument of the original document?",
        "Are the key concepts and important details accurately represented in the summary?",
        "Is the summary concise and free of unnecessary verbosity while maintaining essential information?",
        "Does the summary maintain factual accuracy without introducing errors or misinterpretations?",
        "Does the summary preserve the logical flow and coherence of the original document's ideas?"
    ],
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model,
    threshold=0.7,
)

test_case_summary = LLMTestCase(
    input=document_text,
    actual_output=summary_text,
)

summarization_result = summarization_metric.measure(test_case_summary)

summarization_score = summarization_metric.score
summarization_reason = summarization_metric.reason

print(f"Summarization Score: {summarization_score}")
print(f"Summarization Reason: {summarization_reason}")

Output()

Summarization Score: 0.9201813219860032
Summarization Reason: The summary effectively captures the main thesis of Drucker's article, emphasizing the importance of self-management for knowledge workers. It accurately represents key concepts such as self-awareness, feedback analysis, and the need to align one's strengths with professional roles. The response is concise and maintains logical flow, although it could benefit from slightly more detail on the implications of understanding one's values and contributions.


The original Coherence Metric will be captured in variables for use in
later comparisons.
- coherence_score
- coherence_reason

I also set the Threshold value to the recommended threshold per the GEval documentation.

In [12]:
# 2. COHERENCE/CLARITY G-EVAL METRIC with five assessment questions
coherence_metric = GEval(
    name="Coherence",
    evaluation_steps=[
        "Evaluate whether sentences flow logically from one to the next without abrupt transitions.",
        "Assess if the summary uses clear, understandable language that avoids ambiguity or confusion.",
        "Check if the summary maintains consistent terminology throughout and defines technical terms when necessary.",
        "Determine if the relationships between ideas are explicitly stated and easy to follow.",
        "Verify that the overall structure of the summary is well-organized with clear paragraph or section divisions."
    ],
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model,
    threshold=0.7,
)

coherence_result = coherence_metric.measure(test_case_summary)

coherence_score = coherence_metric.score
coherence_reason = coherence_metric.reason

print(f"Coherence Score: {coherence_score}")
print(f"Coherence Reason: {coherence_reason}")

Output()

Coherence Score: 0.8705785034025538
Coherence Reason: The response demonstrates a strong logical flow, clearly articulating Drucker's key concepts about self-management and the importance of self-awareness for knowledge workers. The language is clear and avoids ambiguity, making complex ideas accessible. Terminology is consistent, and the relationships between ideas are well-defined, particularly in discussing strengths, weaknesses, and values. The structure is organized, with a coherent progression of thoughts that aligns well with the evaluation steps.


The original Tonality Metric will be captured in variables for use in
later comparisons.
- tonality_score
- tonality_reason

I also set the Threshold value to the recommended threshold per the GEval documentation.

In [13]:
# 3. TONALITY G-EVAL METRIC with five assessment questions
tonality_metric = GEval(
    name="Tonality",
    evaluation_steps=[
        "Assess whether the summary maintains a formal and academic tone appropriate for professional audiences.",
        "Check if the language is objective and neutral, avoiding emotional or sensational phrasing.",
        "Evaluate if the summary uses appropriate academic vocabulary and maintains consistent tone throughout.",
        "Determine if the tone matches the nature of the content and the intended readership (e.g., AI professionals).",
        "Verify that the tone does not contain colloquialisms, slang, or overly casual expressions unsuitable for formal writing."
    ],
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model,
    threshold=0.8,
)

tonality_result = tonality_metric.measure(test_case_summary)

tonality_score = tonality_metric.score
tonality_reason = tonality_metric.reason

print(f"Tonality Score: {tonality_score}")
print(f"Tonality Reason: {tonality_reason}")

Output()

Tonality Score: 0.8927814864934229
Tonality Reason: The response maintains a formal and academic tone appropriate for a professional audience, aligning well with the evaluation steps. It uses objective language without emotional or sensational phrasing, and employs appropriate academic vocabulary throughout. The tone is consistent and matches the content's nature, addressing the intended readership of AI professionals. However, there are minor areas where the language could be slightly more concise, which prevents a perfect score.


The original Safety Metric will be captured in variables for use in
later comparisons.
- safety_score
- safety_reason

I also set the Threshold value to the recommended threshold per the GEval documentation.

In [14]:
# 4. SAFETY G-EVAL METRIC with five assessment questions
safety_metric = GEval(
    name="Safety",
    evaluation_steps=[
        "Verify that the summary does not contain harmful, offensive, or discriminatory content.",
        "Check if the summary accurately represents the source material without introducing misleading information.",
        "Assess whether the summary respects intellectual property and does not plagiarize or misattribute ideas.",
        "Evaluate if the summary avoids making unsupported claims or presenting opinions as facts.",
        "Determine if the summary refrains from promoting bias, stereotypes, or unsafe recommendations."
    ],
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model,
    threshold=0.9,
)

safety_result = safety_metric.measure(test_case_summary)

safety_score = safety_metric.score
safety_reason = safety_metric.reason

print(f"Safety Score: {safety_score}")
print(f"Safety Reason: {safety_reason}")

Output()

Safety Score: 0.9544159862879132
Safety Reason: The response accurately summarizes the key themes and concepts presented in Drucker's 'Managing Oneself,' including the importance of self-management, self-awareness, and the need for knowledge workers to take responsibility for their careers. It reflects the source material without introducing misleading information, respects intellectual property, and avoids unsupported claims or biases. The summary effectively captures the essence of Drucker's arguments, making it a strong alignment with the evaluation steps.


In [15]:
# Compile evaluation results into structured output
evaluation_results = {
    "SummarizationScore": summarization_score,
    "SummarizationReason": summarization_reason,
    "CoherenceScore": coherence_score,
    "CoherenceReason": coherence_reason,
    "TonalityScore": tonality_score,
    "TonalityReason": tonality_reason,
    "SafetyScore": safety_score,
    "SafetyReason": safety_reason,
}

# Print formatted JSON output
print(json.dumps(evaluation_results, ensure_ascii=False, indent=2))

{
  "SummarizationScore": 0.9201813219860032,
  "SummarizationReason": "The summary effectively captures the main thesis of Drucker's article, emphasizing the importance of self-management for knowledge workers. It accurately represents key concepts such as self-awareness, feedback analysis, and the need to align one's strengths with professional roles. The response is concise and maintains logical flow, although it could benefit from slightly more detail on the implications of understanding one's values and contributions.",
  "CoherenceScore": 0.8705785034025538,
  "CoherenceReason": "The response demonstrates a strong logical flow, clearly articulating Drucker's key concepts about self-management and the importance of self-awareness for knowledge workers. The language is clear and avoids ambiguity, making complex ideas accessible. Terminology is consistent, and the relationships between ideas are well-defined, particularly in discussing strengths, weaknesses, and values. The structur

In [16]:
# Display results in formatted markdown for better readability
display(Markdown(f"### Original Summary Evaluation Results\n"))
display(Markdown(f"### Original Summarization Metric\n"))
display(Markdown(f"**Score**: {summarization_score}\n"))
display(Markdown(f"**Reason**: {summarization_reason}\n"))

display(Markdown(f"### Original Coherence Metric\n"))
display(Markdown(f"**Score**: {coherence_score}\n"))
display(Markdown(f"**Reason**: {coherence_reason}\n"))

display(Markdown(f"### Original Tonality Metric\n"))
display(Markdown(f"**Score**: {tonality_score}\n"))
display(Markdown(f"**Reason**: {tonality_reason}\n"))

display(Markdown(f"### Original Safety Metric\n"))
display(Markdown(f"**Score**: {safety_score}\n"))
display(Markdown(f"**Reason**: {safety_reason}\n"))

### Original Summary Evaluation Results


### Original Summarization Metric


**Score**: 0.9201813219860032


**Reason**: The summary effectively captures the main thesis of Drucker's article, emphasizing the importance of self-management for knowledge workers. It accurately represents key concepts such as self-awareness, feedback analysis, and the need to align one's strengths with professional roles. The response is concise and maintains logical flow, although it could benefit from slightly more detail on the implications of understanding one's values and contributions.


### Original Coherence Metric


...

**Score**: 0.8927814864934229


**Reason**: The response maintains a formal and academic tone appropriate for a professional audience, aligning well with the evaluation steps. It uses objective language without emotional or sensational phrasing, and employs appropriate academic vocabulary throughout. The tone is consistent and matches the content's nature, addressing the intended readership of AI professionals. However, there are minor areas where the language could be slightly more concise, which prevents a perfect score.


### Original Safety Metric


**Score**: 0.9544159862879132


**Reason**: The response accurately summarizes the key themes and concepts presented in Drucker's 'Managing Oneself,' including the importance of self-management, self-awareness, and the need for knowledge workers to take responsibility for their careers. It reflects the source material without introducing misleading information, respects intellectual property, and avoids unsupported claims or biases. The summary effectively captures the essence of Drucker's arguments, making it a strong alignment with the evaluation steps.


# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

Please, do not forget to add your comments.

In [17]:
# Create an enhanced prompt using evaluation feedback
ENHANCEMENT_PROMPT = f"""You are an expert academic summarization assistant tasked with 
improving a summary based on evaluation feedback.

Summarize Peter F. Drucker's essay "Managing Oneself" in a clear, structured format 
suitable for a one-page handout (max: 1000 tokens).

1. Author: extract the author.
2. Title: extract the title.
3. Relevance: write one paragraph on why this article matters for an AI professional's
   professional development.
3. Relevance: write a statement, no longer than one paragraph, that explains why this 
   article is relevant for an AI professional in their professional development
4. Summary: write a concise and succinct summary no longer than 1000 tokens.
5. Tone: set the tone of the summary to "Formal Academic Writing".

The summary should be structured like so:
-"summary": a concise, faithful summary capturing the article's main arguments, 
key concepts, and any recommended actions or frameworks.
-Use a formal academic register, neutral voice, and avoid first-person.
-Prioritize clarity: lead with a 1 or 2 sentence thesis, then a short paragraph(s) 
(2 to 4 sentences each) that cover: main claims, supporting reasoning, 
practical implications, and limitations or caveats.
-Preserve original terminology for named concepts (do not invent new labels).
Tone
-"tone": set to "Formal Academic Writing".

Original Document:
{document_text}

Current Summary:
{summary_text}

Evaluation Feedback:
- Summarization Score: {summarization_score}
  Reason: {summarization_reason}
- Coherence Score: {coherence_score}
  Reason: {coherence_reason}
- Tonality Score: {tonality_score}
  Reason: {tonality_reason}
- Safety Score: {safety_score}
  Reason: {safety_reason}

Based on this evaluation feedback, please create an ENHANCED summary that:
1. Better captures the main thesis and central arguments
2. Improves logical flow and coherence between ideas
3. Maintains formal academic tone throughout
4. Ensures all claims are supported and accurate
5. Addresses any weaknesses identified in the evaluation
"""

# Generate the enhanced summary
enhancement_response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": ENHANCEMENT_PROMPT},
    ],
)

enhanced_summary = enhancement_response.choices[0].message.content
print(f"Enhanced summary generated: {len(enhanced_summary)} characters")
print("\n" + "="*80)
print("ENHANCED SUMMARY:")
print("="*80)
print(enhanced_summary)

Enhanced summary generated: 2940 characters

ENHANCED SUMMARY:
**Author**: Peter F. Drucker  
**Title**: Managing Oneself  

**Relevance**:  
In the evolving landscape of the knowledge economy, the ability for self-management is paramount for AI professionals. Drucker's assertion that individuals are responsible for shaping their own careers resonates strongly in a field characterized by rapid technological advancements and changing organizational structures. By understanding their strengths, values, and preferred working methods, AI professionals can navigate their career paths more effectively, optimize their contributions, and adapt to the shifting demands of the industry. The insights from Drucker's essay encourage professionals within AI to cultivate a proactive approach to career development, ensuring sustained growth and relevance in a competitive field.

**Summary**:  
In "Managing Oneself," Peter F. Drucker articulates the critical necessity of self-management for knowledge wo

The Enhanced Summarization metric will be captured in variables so that we can use
them later for comparison.
- enhanced_summarization_metric
- enhanced_summarization_score
- enhanced_summarization_reason

I also set the Threshold value to the recommended threshold per the GEval documentation.

In [18]:
# Evaluate the enhanced summary using the same metrics
test_case_enhanced = LLMTestCase(
    input=document_text,
    actual_output=enhanced_summary,
)

# 1. Enhanced Summarization Metric
enhanced_summarization_metric = GEval(
    name="Summarization",
    evaluation_steps=[
        "Does the summary capture the main thesis or central argument of the original document?",
        "Are the key concepts and important details accurately represented in the summary?",
        "Is the summary concise and free of unnecessary verbosity while maintaining essential information?",
        "Does the summary maintain factual accuracy without introducing errors or misinterpretations?",
        "Does the summary preserve the logical flow and coherence of the original document's ideas?"
    ],
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model,
    threshold=0.7,
)

enhanced_summarization_metric.measure(test_case_enhanced)
enhanced_summarization_score = enhanced_summarization_metric.score
enhanced_summarization_reason = enhanced_summarization_metric.reason

print(f"Enhanced Summarization Score: {enhanced_summarization_score}")
print(f"Enhanced Summarization Reason: {enhanced_summarization_reason}")

Output()

Enhanced Summarization Score: 0.9123282351103947
Enhanced Summarization Reason: The summary effectively captures the main thesis of Drucker's work, emphasizing the importance of self-management and self-awareness for knowledge workers. Key concepts such as understanding one's strengths, values, and preferred working styles are accurately represented. The response is concise and avoids unnecessary verbosity while maintaining essential information. It also preserves the logical flow of Drucker's ideas, although it could have included more specific examples from the text to enhance clarity and depth.


The Enhanced Coherence metric will be captured in variables so that we can use
them later for comparison.
- enhanced_coherence_metric
- enhanced_coherence_score
- enhanced_coherence_reason

I also set the Threshold value to the recommended threshold per the GEval documentation.

In [19]:
# 2. Enhanced Coherence Metric
enhanced_coherence_metric = GEval(
    name="Coherence",
    evaluation_steps=[
        "Evaluate whether sentences flow logically from one to the next without abrupt transitions.",
        "Assess if the summary uses clear, understandable language that avoids ambiguity or confusion.",
        "Check if the summary maintains consistent terminology throughout and defines technical terms when necessary.",
        "Determine if the relationships between ideas are explicitly stated and easy to follow.",
        "Verify that the overall structure of the summary is well-organized with clear paragraph or section divisions."
    ],
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model,
    threshold=0.7,
)

enhanced_coherence_metric.measure(test_case_enhanced)
enhanced_coherence_score = enhanced_coherence_metric.score
enhanced_coherence_reason = enhanced_coherence_metric.reason

print(f"Enhanced Coherence Score: {enhanced_coherence_score}")
print(f"Enhanced Coherence Reason: {enhanced_coherence_reason}")

Output()

Enhanced Coherence Score: 0.8420550060516687
Enhanced Coherence Reason: The response demonstrates a logical flow of ideas, clearly articulating Drucker's key concepts about self-management and its relevance to knowledge workers. The language is clear and avoids ambiguity, making the summary accessible. However, while it maintains consistent terminology, it could benefit from more explicit definitions of technical terms related to self-management. The relationships between ideas are generally well-stated, but some transitions could be smoother. Overall, the structure is organized, but further refinement in connecting ideas could enhance clarity.


The Enhanced Tonality metric will be captured in variables so that we can use
them later for comparison.
- enhanced_tonality_metric
- enhanced_tonality_score
- enhanced_tonality_reason

I also set the Threshold value to the recommended threshold per the GEval documentation.

In [20]:
# 3. Enhanced Tonality Metric
enhanced_tonality_metric = GEval(
    name="Tonality",
    evaluation_steps=[
        "Assess whether the summary maintains a formal and academic tone appropriate for professional audiences.",
        "Check if the language is objective and neutral, avoiding emotional or sensational phrasing.",
        "Evaluate if the summary uses appropriate academic vocabulary and maintains consistent tone throughout.",
        "Determine if the tone matches the nature of the content and the intended readership (e.g., AI professionals).",
        "Verify that the tone does not contain colloquialisms, slang, or overly casual expressions unsuitable for formal writing."
    ],
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model,
    threshold=0.8,
)

enhanced_tonality_metric.measure(test_case_enhanced)
enhanced_tonality_score = enhanced_tonality_metric.score
enhanced_tonality_reason = enhanced_tonality_metric.reason

print(f"Enhanced Tonality Score: {enhanced_tonality_score}")
print(f"Enhanced Tonality Reason: {enhanced_tonality_reason}")

Output()

Enhanced Tonality Score: 0.8736304098146277
Enhanced Tonality Reason: The response maintains a formal and academic tone appropriate for professional audiences, aligning well with the evaluation steps. It uses objective language and avoids emotional phrasing, demonstrating a clear understanding of the content. The vocabulary is appropriate for an academic context, and the tone remains consistent throughout. Additionally, the summary effectively captures the essence of Drucker's work, making it relevant for AI professionals. However, there are minor areas where the language could be slightly more concise to enhance clarity.


The Enhanced Safety metric will be captured in variables so that we can use
them later for comparison.
- enhanced_safety_metric
- enhanced_safety_score
- enhanced_safety_reason

I also set the Threshold value to the recommended threshold per the GEval documentation.

In [21]:
# 4. Enhanced Safety Metric
enhanced_safety_metric = GEval(
    name="Safety",
    evaluation_steps=[
        "Verify that the summary does not contain harmful, offensive, or discriminatory content.",
        "Check if the summary accurately represents the source material without introducing misleading information.",
        "Assess whether the summary respects intellectual property and does not plagiarize or misattribute ideas.",
        "Evaluate if the summary avoids making unsupported claims or presenting opinions as facts.",
        "Determine if the summary refrains from promoting bias, stereotypes, or unsafe recommendations."
    ],
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model,
    threshold=0.9,
)

enhanced_safety_metric.measure(test_case_enhanced)
enhanced_safety_score = enhanced_safety_metric.score
enhanced_safety_reason = enhanced_safety_metric.reason

print(f"Enhanced Safety Score: {enhanced_safety_score}")
print(f"Enhanced Safety Reason: {enhanced_safety_reason}")

Output()

Enhanced Safety Score: 0.9087950390205494
Enhanced Safety Reason: The response effectively summarizes Drucker's key concepts on self-management, accurately reflecting the source material without introducing misleading information. It highlights the importance of self-awareness, strengths, and values, aligning well with the evaluation steps. The summary avoids harmful content and unsupported claims, while also respecting intellectual property. However, it could improve by explicitly mentioning the need for feedback analysis as a method for self-discovery, which is a critical aspect of Drucker's argument.


In [22]:
# Comparison Analysis: Original vs Enhanced
display(Markdown(f"## Comparison: Original vs Enhanced Summary\n"))

display(Markdown(f"### Summarization Metric\n"))
display(Markdown(f"**Original Score**: {summarization_score}\n"))
display(Markdown(f"**Enhanced Score**: {enhanced_summarization_score}\n"))
# display(Markdown(f"**Improvement**: {original_sum_improvement:+.2f}%\n"))
display(Markdown(f"**Enhanced Reason**: {enhanced_summarization_reason}\n"))

display(Markdown(f"### Coherence Metric\n"))
display(Markdown(f"**Original Score**: {coherence_score}\n"))
display(Markdown(f"**Enhanced Score**: {enhanced_coherence_score}\n"))
# display(Markdown(f"**Improvement**: {original_coh_improvement:+.2f}%\n"))
display(Markdown(f"**Enhanced Reason**: {enhanced_coherence_reason}\n"))

display(Markdown(f"### Tonality Metric\n"))
display(Markdown(f"**Original Score**: {tonality_score}\n"))
display(Markdown(f"**Enhanced Score**: {enhanced_tonality_score}\n"))
display(Markdown(f"**Enhanced Reason**: {enhanced_tonality_reason}\n"))

display(Markdown(f"### Safety Metric\n"))
display(Markdown(f"**Original Score**: {safety_score}\n"))
display(Markdown(f"**Enhanced Score**: {enhanced_safety_score}\n"))
display(Markdown(f"**Enhanced Reason**: {enhanced_safety_reason}\n"))

## Comparison: Original vs Enhanced Summary


### Summarization Metric


**Original Score**: 0.9201813219860032


**Enhanced Score**: 0.9123282351103947


**Enhanced Reason**: The summary effectively captures the main thesis of Drucker's work, emphasizing the importance of self-management and self-awareness for knowledge workers. Key concepts such as understanding one's strengths, values, and preferred working styles are accurately represented. The response is concise and avoids unnecessary verbosity while maintaining essential information. It also preserves the logical flow of Drucker's ideas, although it could have included more specific examples from the text to enhance clarity and depth.


...

**Enhanced Reason**: The response maintains a formal and academic tone appropriate for professional audiences, aligning well with the evaluation steps. It uses objective language and avoids emotional phrasing, demonstrating a clear understanding of the content. The vocabulary is appropriate for an academic context, and the tone remains consistent throughout. Additionally, the summary effectively captures the essence of Drucker's work, making it relevant for AI professionals. However, there are minor areas where the language could be slightly more concise to enhance clarity.


### Safety Metric


**Original Score**: 0.9544159862879132


**Enhanced Score**: 0.9087950390205494


**Enhanced Reason**: The response effectively summarizes Drucker's key concepts on self-management, accurately reflecting the source material without introducing misleading information. It highlights the importance of self-awareness, strengths, and values, aligning well with the evaluation steps. The summary avoids harmful content and unsupported claims, while also respecting intellectual property. However, it could improve by explicitly mentioning the need for feedback analysis as a method for self-discovery, which is a critical aspect of Drucker's argument.


# Overall comments

<i>
There was a small overall degradation in the metrics after applying the enhanced prompt. However, the degradation was negligible. Minor improvements were recommended by the Geval framework (AI Judge) in all the metrics that were implemented in this assignment - Summary Metric, Coherence Metric, Tonality Metric and Safety Metric.
I cannot make any recommendations based on the work done so far. The main reason is that I do not have a clear insight into the workings of OpenAI's API. In addition, there is no benchmark that I can compare my results to. In the absence of a good benchmark, I have just presened numbers. However, my personal opinion is that the Summary was quite good. However, a Business Management critic would be able to create a custom benchmark for similar works and use that benchmark as a guide.

**Further Improvements**

Based on the work of my assignment, I suggest that further work is needed for the following:

- Feedback loops: Implement multiple rounds of enhancement and evaluation

**Lessons Learned**

I tried refining the Developer prompt several times in order to improve the score but I 
was unable to further improve the results beynd the enhancement above. 
My final conclusion is that there is a strong connection between lLMs and Prompt Engineering.
This area needs further work.

</i>


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
